# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Data-access blocker: the paper's two required Dryad CSVs (doi:10.5061/dryad.5x69p8djr v3, CC0; files 4577414 and 4577415) are unreachable through authorized programmatic routes. Executed probes on a Colab VM (logs/probe1-probe5) show: REST API download endpoints (per-file and dataset-level) return 401 'Unauthorized, must have current bearer token'; the website file_stream route serves an Anubis anti-bot JS challenge to browser UAs and 403 to tool UAs (automation prohibited); no public mirror exists (GitHub/Zenodo/Wayback searched). Code (PhoTorch @ e6e4265) and paper identity/Eqn 4/Table 1 were fully verified from primary sources. Per outcome B, deliverables/p-6dbbe4f1f990f82f9de373035b88c898.ipynb is a complete CODE-ONLY / NOT EXECUTED notebook (statically validated, dated access guard, no fabricated outputs); eligible for the host queue's UNVERIFIED-reference path. Needs a permitted access path or public mirror before fresh validation.

Attempt status: blocked. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Leaf gas exchange measurement for steady‐state stomatal conductance model calibration

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-6dbbe4f1f990f82f9de373035b88c898)


# Steady-state stomatal conductance model calibration — BTA/BMF refit with PhoTorch

**Paper:** Rizzo, K.T. & Bailey, B.N. (2026). *Leaf gas exchange measurement for steady-state stomatal conductance model calibration.* New Phytologist 250(1):672–690. DOI [10.1111/nph.70949](https://doi.org/10.1111/nph.70949) (PMC12961262).

**Scope of this notebook:** refit the paper's Buckley–Turnbull–Adams (BTA) steady-state stomatal conductance model (Eqn 4) to the authors' own steady-state IRGA response data, using the authors' PhoTorch package, and compare the extracted semi-parameters **Em, i0, k, b** against Table 1 (six single-leaf steady-state IRGA fits, English walnut) and the paper's steady-state response-curve fit (Em 21.88 mmol m⁻² s⁻¹, i0 11.47 µmol m⁻² s⁻¹, k 138 700 µmol m⁻² s⁻¹ mmol mol⁻¹, b 0).

**⚠️ STATUS: CODE-ONLY / NOT EXECUTED — DATA ACCESS BLOCKED (2026-10-25).**
The required input files are hosted on Dryad (doi:10.5061/dryad.5x69p8djr, v3, CC0). As of 2026-10-25 every authorized programmatic public route returned an access control, verified by executed probes (see report):
- `GET https://datadryad.org/api/v2/files/{id}/download` → **401** `{"error":"Unauthorized, must have current bearer token"}` (also for the dataset-level `/download`);
- the website's `https://datadryad.org/stash/downloads/file_stream/{id}` → Anubis anti-bot JavaScript challenge for browser User-Agents and **403 Forbidden** for plain tool User-Agents (solving/automating the challenge is not permitted);
- no public mirror was found (GitHub/Zenodo searched; Wayback CDX empty for these URLs).

**⚠️ ステータス: コードのみ / 未実行 — データアクセスがブロック(2026-10-25)。** 必須入力は Dryad 限定で、API ダウンロードはトークン必須(401)、ウェブ経由はアンチボット検証(403/JSチャレンジ)のため、検証済み実行が不可能です。アクセス条件が解決した場合のみ、このノートブックは先頭から実行できます。

A faithful demonstration is impossible without the paper's data bytes, so this notebook was never executed and contains no fabricated results. Every code cell is preceded by an explanation. If the access condition is ever resolved (e.g., Dryad restores anonymous programmatic downloads or a public mirror appears), the notebook below is complete and ready to run top-to-bottom on a **CPU** runtime.

**実行計画:** CPU のみ(SGD フィット、データは小規模)。GPU 不要。

### Guard — verify the paper data are actually reachable before anything else

The first code cell downloads the smaller required input (`Figure6-Steady-State-Measured.csv`, Dryad file 4577414) through the documented public routes and checks its published SHA-256 (`3bc02eb4…c0dc8`, size 1 195 B). If neither route yields the verified bytes it raises `RuntimeError` with the precise access condition, so *Run all* can never silently produce a misleading result.

**日本語:** 最初のセルで必須入力の到達性と SHA-256 を検証し、取得できない場合は明確な理由付きで停止します(誤った結果を出力させないためのガード)。

In [ ]:
import hashlib, io, requests

DRYAD_FILE_URLS = {
    "Figure6-Steady-State-Measured.csv": "https://datadryad.org/api/v2/files/4577414/download",
}
FILE_STREAM_FALLBACK = "https://datadryad.org/stash/downloads/file_stream/4577414"
EXPECTED_SHA256 = "3bc02eb47c676d6dc1ed8c3c3b5be3bf168b95b872974eef2c7ddd19252c0dc8"
EXPECTED_SIZE = 1195
UA = {"User-Agent": "python-requests (research reproduction of DOI 10.1111/nph.70949)"}

blob = None
for url in (DRYAD_FILE_URLS["Figure6-Steady-State-Measured.csv"], FILE_STREAM_FALLBACK):
    r = requests.get(url, headers=UA, timeout=180)
    print(url.rsplit("/", 2)[-3], "->", r.status_code, r.headers.get("content-type"))
    if r.status_code == 200 and "html" not in r.headers.get("content-type", ""):
        blob = r.content
        break

if blob is None or hashlib.sha256(blob).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError(
        "DATA ACCESS BLOCKED (checked 2026-10-25): the paper's Dryad files "
        "(doi:10.5061/dryad.5x69p8djr, v3) are not reachable through authorized "
        "programmatic routes. The Dryad REST API download endpoints return 401 "
        "'Unauthorized, must have current bearer token'; the website "
        "file_stream route serves an Anubis anti-bot challenge to browser UAs "
        "and 403 to tool UAs; no public mirror exists. A Dryad API account "
        "(manual registration) or a public mirror would be required. "
        "See report.md for the full probe record."
    )
open("/content/Figure6-Steady-State-Measured.csv", "wb").write(blob)
print("Guard passed: Figure6 CSV verified,", len(blob), "bytes")

## 1. Runtime selection — CPU

**Runtime: CPU only.** The reproduction is two Adam stochastic-gradient fits over at most ~2 000 tabular points with four parameters per curve; the paper's PhoTorch README example itself defaults to `device_fit='cpu'`. A GPU gives no scientific benefit here, so this notebook requests no accelerator.

**日本語:** 本デモは CPU のみで十分です(数千点規模のテーブルデータへの Adam 最適化)。GPU は不要です。

### Install the authors' PhoTorch at the pinned revision

This installs `photorch` directly from the authors' GitHub repository at the commit verified for this paper (`e6e4265717be650c88a9fa14f5748d52b3b8de9a`, setup.py version 1.4.7, GPL-3.0). Dependencies (`torch`, `numpy`, `pandas`, `scipy`, `matplotlib`) resolve automatically; the printout confirms the exact import.

**日本語:** 論文の著者らの PhoTorch を、検証済みコミットに固定してインストールします。

In [ ]:
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/GEMINI-Breeding/photorch@e6e4265717be650c88a9fa14f5748d52b3b8de9a"],
    check=True,
)
import photorch, torch, pandas as pd, numpy as np
from photorch import stomatal, util
print("photorch import OK | torch", torch.__version__, "| pandas", pd.__version__,
      "| numpy", np.__version__)

### Download both required Dryad files with checksum verification

The two minimal inputs for the reproduction (Dryad version record 422044, CC0, published 2026-01-29 by Rizzo & Bailey):
- `Figure6-Steady-State-Measured.csv` — 1 195 B, SHA-256 `3bc02eb4…c0dc8` (file 4577414): the steady-state measured response data of Fig. 6;
- `Figure7-IRGA-Single-Leaves.csv` — 60 797 B, SHA-256 `a73cc860…d97f` (file 4577415): the six single-leaf steady-state IRGA datasets of Fig. 7 / Table 1.

Each file is fetched via the Dryad REST API (the documented machine route), with the website `file_stream` endpoint as fallback, and its size + SHA-256 are asserted before use.

**日本語:** 再現に必要な 2 つの CSV を Dryad から取得し、サイズと SHA-256 を検証してから保存します。

In [ ]:
import hashlib, requests

FILES = {
    "Figure6-Steady-State-Measured.csv": (4577414, 1195,
        "3bc02eb47c676d6dc1ed8c3c3b5be3bf168b95b872974eef2c7ddd19252c0dc8"),
    "Figure7-IRGA-Single-Leaves.csv": (4577415, 60797,
        "a73cc860fc3f617ea6d7db270933bd3611903b7ae533d86e7b044c1d38c9d97f"),
}
UA = {"User-Agent": "python-requests (research reproduction of DOI 10.1111/nph.70949)"}

for name, (fid, size, sha) in FILES.items():
    blob = None
    for url in (f"https://datadryad.org/api/v2/files/{fid}/download",
                f"https://datadryad.org/stash/downloads/file_stream/{fid}"):
        r = requests.get(url, headers=UA, timeout=300)
        print(name, "<-", url.rsplit('/', 2)[-3], r.status_code, r.headers.get("content-type"))
        if r.status_code == 200 and "html" not in r.headers.get("content-type", ""):
            blob = r.content
            break
    assert blob is not None, f"{name}: no route returned the file"
    assert len(blob) == size, f"{name}: size {len(blob)} != expected {size}"
    assert hashlib.sha256(blob).hexdigest() == sha, f"{name}: SHA-256 mismatch"
    open("/content/" + name, "wb").write(blob)
    print(f"verified {name}: {len(blob)} bytes, SHA-256 OK")

### Inspect the actual input schema before fitting anything

Column layout, CurveID values, units and record counts of both CSVs are printed here **before** any model code, so no column name, curve count, or CurveID→leaf mapping is ever hard-coded from assumption. PhoTorch's `initscdata` requires `CurveID`, `gsw`, `Tleaf`, `VPDleaf`, and either (`RHcham`, `Qin`) or (`rh_r`, `Qamb`); `A` is optional for the BMF model.

**日本語:** フィット前に CSV の列構成・CurveID・行数を確認します(列名や対応関係を仮定で決め打ちしないため)。

In [ ]:
df6 = pd.read_csv("/content/Figure6-Steady-State-Measured.csv")
df7 = pd.read_csv("/content/Figure7-IRGA-Single-Leaves.csv")

for label, df in (("Figure6", df6), ("Figure7", df7)):
    print(f"== {label}: {df.shape[0]} rows x {df.shape[1]} cols")
    print(df.dtypes.to_string())
    print(df.head(3).to_string(), "\n")

required = {"CurveID", "gsw", "Tleaf", "VPDleaf"}
for label, df in (("Figure6", df6), ("Figure7", df7)):
    missing = required - set(df.columns)
    assert not missing, f"{label}: PhoTorch-required columns missing: {missing}"
    if "CurveID" not in df.columns or df["CurveID"].isna().all():
        df["CurveID"] = 0
    print(label, "CurveID value counts:", df["CurveID"].value_counts().sort_index().to_dict())

### Input preview — measured g_sw against light and VPD

Before any fitting, the actual measured data are plotted: stomatal conductance versus absorbed PPFD and versus VPD, one marker series per CurveID. This is the raw input the model will be fitted to (directly observed content, not model output).

**日本語:** フィット前に、測定された気孔コンダクタンスを光(Q)と VPD(D)に対して可視化します。

In [ ]:
import matplotlib.pyplot as plt

D_CONV = 1000.0 / 101.3  # kPa -> mmol/mol (PhoTorch initscdata conversion)
Q_ABS = 0.85             # leaf PAR absorptivity assumed by PhoTorch initscdata

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for label, df in (("Fig6", df6), ("Fig7", df7)):
    for cid, g in df.groupby("CurveID"):
        axes[0].scatter(g["Qin"] * Q_ABS, g["gsw"], s=18, alpha=0.7,
                        label=f"{label} CurveID {cid}")
        axes[1].scatter(g["VPDleaf"] * D_CONV, g["gsw"], s=18, alpha=0.7)
axes[0].set_xlabel(r"Absorbed PPFD $Q$ ($\mu$mol m$^{-2}$ s$^{-1}$)")
axes[0].set_ylabel(r"Measured g$_{sw}$ (mol m$^{-2}$ s$^{-1}$)")
axes[0].legend(fontsize=7)
axes[1].set_xlabel(r"$D$ (mmol mol$^{-1}$)")
axes[1].set_ylabel(r"Measured g$_{sw}$ (mol m$^{-2}$ s$^{-1}$)")
fig.suptitle("Input preview: steady-state IRGA measurements (Dryad, verified SHA-256)")
fig.tight_layout()
plt.show()

## 2. Core method — BTA/BMF model and PhoTorch fitting

The paper's Eqn 4 (BTA; verified from the Europe PMC full text of PMC12961262) is

**g_sw = Em · (Q + i0) / (k + b·Q + (Q + i0)·D)**

with Q the absorbed PPFD (µmol m⁻² s⁻¹) and D = VPD/p_atm (mmol mol⁻¹). PhoTorch's `BMF` class (`photorch/src/stomatal/stomatalmodels.py`, pinned commit) implements exactly this equation, and its `initscdata` applies the same unit conversions (Q = 0.85·Qin; D = VPDleaf·1000/101.3). The loss adds the paper's biological constraints (non-negative parameters, positive light response with negative curvature, negative VPD response — paper §Model fitting). Fitting follows the authors' README defaults: `learnrate=0.5`, `maxiteration=20000` (Adam, StepLR ×0.9 every 5 000 steps).

**日本語:** 論文 Eqn 4(BTA)は PhoTorch の BMF と同一の式であり、著者のデフォルト設定(Adam, 20 000 反復)でフィットします。

### Fit the Figure 6 steady-state measured response data

All rows are treated as one curve (CurveID forced to 0 above if absent). The printed parameters are compared with the paper's steady-state response-curve fit (Em 21.88, i0 11.47, k 138 700, b 0 — flow phase *Simulated data generation*, quoting the paper). Small deviations are expected: the paper does not state iteration counts, and the b = 0 boundary is enforced only softly through the penalty loss.

**日本語:** Fig. 6 の定常応答データを 1 曲線としてフィットし、論文の応答曲線フィット値(Em 21.88, i0 11.47, k 138 700, b 0)と比較します。

In [ ]:
df6_fit = df6.copy()
if "CurveID" not in df6_fit.columns or df6_fit["CurveID"].isna().all():
    df6_fit["CurveID"] = 0

scd6 = stomatal.initscdata(df6_fit)
scm6 = stomatal.BMF(scd6)
fit6 = stomatal.fit(scm6, learnrate=0.5, maxiteration=20000)
scm6 = fit6.model

PAPER_RESPONSE_CURVE = {"Em": 21.88, "i0": 11.47, "k": 138700.0, "b": 0.0}
rows = []
for i, cid in enumerate(scd6.IDs):
    got = {"Em": scm6.Em[i].item(), "i0": scm6.i0[i].item(),
           "k": scm6.k[i].item(), "b": scm6.b[i].item()}
    rows.append({"CurveID": int(cid), **{p: round(got[p], 3) for p in got},
                 **{"paper_" + p: PAPER_RESPONSE_CURVE[p] for p in got}})
pd.DataFrame(rows)

### Fit the six Figure 7 / Table 1 single-leaf steady-state IRGA datasets

Each `CurveID` in `Figure7-IRGA-Single-Leaves.csv` is one leaf: PhoTorch assigns an independent parameter set per CurveID, which reproduces the paper's per-leaf fitting (Table 1, steady-state IRGA rows). The CurveID→Leaf mapping is read from the schema-inspection cell output and stated in the comparison table; it is not hard-coded.

**日本語:** Fig. 7 の 6 葉分データを CurveID ごとに独立フィットします(論文 Table 1 の定常・IRGA 行に対応)。

In [ ]:
scd7 = stomatal.initscdata(df7)
scm7 = stomatal.BMF(scd7)
fit7 = stomatal.fit(scm7, learnrate=0.5, maxiteration=20000)
scm7 = fit7.model

from photorch.src.util import computeR2  # author's R^2 implementation

records = []
for i, cid in enumerate(scd7.IDs):
    idx = scd7.getIndicesbyID(cid)
    gsw_pred = scm7()[idx].detach().numpy()
    gsw_obs = scd7.gsw[idx].detach().numpy()
    records.append({"CurveID": int(cid),
                    "Em": scm7.Em[i].item(), "i0": scm7.i0[i].item(),
                    "k": scm7.k[i].item(), "b": scm7.b[i].item(),
                    "R2": computeR2(gsw_obs, gsw_pred),
                    "n_points": len(idx)})
fitted7 = pd.DataFrame(records).round(3)
fitted7

### Compare fitted parameters with the paper's Table 1 (steady-state IRGA rows)

The reference values below are transcribed from Table 1 of the paper's verified full text (New Phytologist 250(1):672–690): per-leaf Em, i0, k, b and R². Rows are joined on the CurveID→Leaf mapping observed in the schema-inspection output; if the CSV ordering does not match the Table 1 leaf numbering, the mapping row must be corrected here before interpreting differences.

**日本語:** フィット結果を論文 Table 1(定常・IRGA)の値と比較します。CurveID と葉番号の対応はスキーマ確認セルの出力に基づきます。

In [ ]:
# Table 1, steady-state IRGA rows (New Phytologist 250(1):672-690), units:
# Em mmol m^-2 s^-1; i0 umol m^-2 s^-1; k umol m^-2 s^-1 mmol mol^-1; b mmol mol^-1
TABLE1_STEADY_IRGA = {
    1: {"Em": 10.6,  "i0": 772.0, "k": 2.88e5, "b": 46.1, "R2": 0.985},
    2: {"Em": 4.18,  "i0": 244.0, "k": 2.99e4, "b": 11.0, "R2": 0.982},
    3: {"Em": 4.96,  "i0": 228.0, "k": 6.19e4, "b": 16.0, "R2": 0.978},
    4: {"Em": 3.07,  "i0": 77.7,  "k": 9.90e3, "b": 2.03, "R2": 0.982},
    5: {"Em": 3.81,  "i0": 1.22,  "k": 7.53e3, "b": 0.51, "R2": 0.990},
    6: {"Em": 3.24,  "i0": 159.0, "k": 4.02e4, "b": 1.23, "R2": 0.951},
}

# Observed CurveID -> Table 1 Leaf mapping (confirm against the schema-inspection
# output above; adjust if the CSV's CurveID order differs from Table 1 leaf order).
curves_in_order = list(fitted7["CurveID"])
assert len(curves_in_order) == len(TABLE1_STEADY_IRGA), (
    f"Expected {len(TABLE1_STEADY_IRGA)} curves, CSV has {len(curves_in_order)}")
curve_to_leaf = {cid: leaf for cid, leaf in zip(curves_in_order, sorted(TABLE1_STEADY_IRGA))}

comparison = fitted7.copy()
comparison["Leaf(Table1)"] = comparison["CurveID"].map(curve_to_leaf)
for p in ("Em", "i0", "k", "b"):
    comparison[f"paper_{p}"] = comparison["Leaf(Table1)"].map(lambda l: TABLE1_STEADY_IRGA[l][p])
    comparison[f"rel.diff_{p}"] = ((comparison[p] - comparison[f"paper_{p}"])
                                   / comparison[f"paper_{p}"]).abs().round(3)
comparison["paper_R2"] = comparison["Leaf(Table1)"].map(lambda l: TABLE1_STEADY_IRGA[l]["R2"])
comparison

### Visualize per-leaf measured data and fitted BTA response curves

For each leaf: measured points (markers) and the fitted BMF model evaluated along a fixed light×VPD slice through each measured point's complementary coordinate — an additional visualization created for this notebook, not an author figure. The saved PNG also serves as the reproducible result record.

**日本語:** 各葉について、実測点とフィット済み BMF モデル曲線を重ねて描画します(本ノートブック独自の追加可視化であり、論文図の再現ではないことに注意)。

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False)
for ax, (_, row) in zip(axes.ravel(), comparison.iterrows()):
    cid = row["CurveID"]
    idx = scd7.getIndicesbyID(cid)
    g = df7[df7["CurveID"] == cid]
    Qg = g["Qin"].to_numpy() * Q_ABS
    Dg = g["VPDleaf"].to_numpy() * D_CONV
    qs = np.linspace(max(Qg.min() * 0.9, 1.0), Qg.max() * 1.1, 120)
    D_med = np.median(Dg)
    gs_curve = row["Em"] * (qs + row["i0"]) / (row["k"] + row["b"] * qs + (qs + row["i0"]) * D_med)
    ax.scatter(Qg, g["gsw"], s=22, label="measured (fixed D slice)")
    ax.plot(qs, gs_curve, "r-", label=f"BMF fit (D={D_med:.0f} mmol/mol)")
    ax.set_title(f"CurveID {cid} → Leaf {row['Leaf(Table1)']} | R2 {row['R2']:.3f} (paper {row['paper_R2']:.3f})",
                 fontsize=9)
    ax.set_xlabel(r"$Q$ ($\mu$mol m$^{-2}$ s$^{-1}$)")
    ax.set_ylabel(r"g$_{sw}$ (mol m$^{-2}$ s$^{-1}$)")
    ax.legend(fontsize=7)
fig.suptitle("BTA/BMF per-leaf fits vs. measured steady-state IRGA data (Fig 7 / Table 1)")
fig.tight_layout()
plt.savefig("/content/fig7_bmf_fits.png", dpi=150)
plt.show()

### Export the comparison table

The fitted-vs-paper comparison is written to CSV inside the Colab VM for inspection or download.

**日本語:** 比較表を Colab 内に CSV として保存します。

In [ ]:
comparison.to_csv("/content/table1_comparison.csv", index=False)
print(open("/content/table1_comparison.csv").read())

## 3. Interpretation, differences from the paper, and validation record

- **What this notebook would establish:** whether the authors' released PhoTorch code, run on the authors' released steady-state IRGA data, recovers the Table 1 per-leaf BTA parameters and the response-curve fit reported in the paper — an independent re-execution of the paper's calibration path (paper §Model fitting; flow phase *BTA parameter fitting with PhoTorch*).
- **Expected differences:** the paper does not state optimizer iterations/learning rate; this notebook uses the PhoTorch README defaults. Table 1 reports three significant figures, and the b = 0 boundary is enforced softly, so small deviations (especially in b for leaves with paper b ≈ 0.5–1) are plausible. The CurveID→Leaf mapping must be confirmed from the schema output.
- **Out of scope** (recorded limitations): porometer/survey fits require the psychrometric correction of Rizzo & Bailey 2025b (documented in a separate publication, not reproducible from this paper alone); BBL comparisons (Table S5) cannot be rerun because the released PhoTorch does not include BBL; the Helios 3D simulation branch requires a C++ build and is omitted.
- **Validation record:** this notebook is CODE-ONLY / NOT EXECUTED (blocked 2026-10-25 by the Dryad access condition documented at the top). No outputs, metrics, or figures in it are evidence; all numeric claims belong to the cited paper. If executed after the access condition is resolved, this section should be replaced by the actual validation record.
- **実行されていないことに注意:** 本ノートブックは未実行であり、数値はすべて引用論文由来です。

### References
1. Rizzo, K.T. & Bailey, B.N. (2026). New Phytologist 250(1):672–690. DOI 10.1111/nph.70949; PMC12961262 (full text via Europe PMC).
2. Data: Dryad doi:10.5061/dryad.5x69p8djr (v3, CC0; files 4577414, 4577415).
3. Code: PhoTorch, GEMINI-Breeding/photorch @ e6e4265717be650c88a9fa14f5748d52b3b8de9a (GPL-3.0), files `photorch/src/stomatal/{initscdata,stomatalmodels,fitstomatal,evaluate}.py`, `photorch/src/util.py`.
4. Lei, T., Rizzo, K.T., & Bailey, B.N. (2025). PhoTorch. Photosynthesis Research 163(2):21. DOI 10.1007/s11120-025-01136-7.